# Khám phá dữ liệu email Enron1
Chỉ khám phá và minh họa các bước; notebook train nằm trong Models/.

In [ ]:
from pathlib import Path

# Tìm thư mục dự án từ vị trí đang mở notebook.
ROOT = Path.cwd().resolve()
while not (ROOT / "Datasets/data_raw/spam_ham_dataset.csv").is_file():
    if ROOT == ROOT.parent:
        raise FileNotFoundError("Hãy mở notebook bên trong thư mục Present_and_Lab1_ML_G5.")
    ROOT = ROOT.parent

print("Thư mục dự án:", ROOT)


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
import re

# Hàm làm sạch viết ngay tại đây, không import từ file khác.
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"<[^>]*>", " ", text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return " ".join(text.split())

# 1. Đọc dữ liệu: chỉ lấy văn bản và nhãn.
data = pd.read_csv(ROOT / "Datasets/data_raw/spam_ham_dataset.csv")
data = data[["text", "label"]].dropna().copy()
print("Trước xử lý:", len(data))

# 2. Làm sạch và loại nội dung trùng trước khi chia tập.
data["clean_text"] = data["text"].apply(clean_text)
data = data[data["clean_text"] != ""]
if data.groupby("clean_text")["label"].nunique().max() > 1:
    raise ValueError("Có nội dung trùng nhưng khác nhãn.")
data = data.drop_duplicates(subset="clean_text")
print("Sau xử lý:", len(data))
print(data["label"].value_counts())
data[["label", "clean_text"]].head()


## Chia tập và vector hóa chỉ từ train

In [ ]:
# 3. Chia văn bản trước khi học TF-IDF.
text_train, text_test, y_train, y_test = train_test_split(
    data["clean_text"], data["label"],
    test_size=0.2, random_state=42, stratify=data["label"]
)
print("Train:", len(text_train), "Test:", len(text_test))


In [ ]:
# 4. Học từ vựng và trọng số từ train, chỉ biến đổi test.
vectorizer = TfidfVectorizer(stop_words="english", max_features=1000)
X_train = vectorizer.fit_transform(text_train)
X_test = vectorizer.transform(text_test)
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)


## Một số từ trong từ vựng học từ train

In [ ]:
print(vectorizer.get_feature_names_out()[:30])